In [1]:

import pandas as pd

# Dataset 1: Inventory
inventory_data = {
    "item_id": ["I001", "I002", "I003", "I004", "I005"],
    "item_name": ["pen", "book", "pencil", "notebook", "eraser"],
    "category": ["Stationery", "Stationery", "Stationery",
                 "Stationery", "Stationery"],
    "quantity": [20, 15, 30, 10, 25],
    "unit_price": [10, 50, 5, 40, 5]
}

inventory_df = pd.DataFrame(inventory_data)
inventory_df.to_csv("inventory_dataset.csv", index=False)

# Dataset 2: Natural-language commands
command_data = {
    "command_text": [
        "Add 10 pens",
        "Remove 3 books",
        "Check stock of pencils",
        "List inventory",
        "Add pens",
        "Fly 5 pens"
    ],
    "intent": ["ADD", "REMOVE", "CHECK", "LIST", "ADD", "UNKNOWN"],
    "quantity": [10, 3, None, None, None, 5],
    "item_name": ["pen", "book", "pencil", None, "pen", "pen"],
    "expected_valid": [True, True, True, True, False, False]
}

commands_df = pd.DataFrame(command_data)
commands_df.to_csv(
    "natural_language_commands_dataset.csv", index=False
)

print("Both datasets created successfully!")

print("\nInventory Dataset:")
display(inventory_df)

print("\nCommand Dataset:")
display(commands_df)

# Download both CSV files as one ZIP
import zipfile

with zipfile.ZipFile("FLA_Inventory_Project_Datasets.zip", "w") as z:
    z.write("inventory_dataset.csv")
    z.write("natural_language_commands_dataset.csv")

print("\nZIP file created successfully!")


Both datasets created successfully!

Inventory Dataset:


,item_id,item_name,category,quantity,unit_price
0,I001,pen,Stationery,20,10
1,I002,book,Stationery,15,50
2,I003,pencil,Stationery,30,5
3,I004,notebook,Stationery,10,40
4,I005,eraser,Stationery,25,5



Command Dataset:


,command_text,intent,quantity,item_name,expected_valid
0,Add 10 pens,ADD,10.0,pen,True
1,Remove 3 books,REMOVE,3.0,book,True
2,Check stock of pencils,CHECK,NaN,pencil,True
3,List inventory,LIST,NaN,None,True
4,Add pens,ADD,NaN,pen,False
5,Fly 5 pens,UNKNOWN,5.0,pen,False



ZIP file created successfully!


In [2]:
import pandas as pd

# Load both datasets
inventory_df = pd.read_csv("inventory_dataset.csv")
commands_df = pd.read_csv("natural_language_commands_dataset.csv")

# Create inventory dictionary from the dataset
inventory = dict(zip(
    inventory_df["item_name"],
    inventory_df["quantity"]
))

print("Inventory loaded successfully!")
print(inventory)

print("\nCommand dataset loaded!")
print("Total test commands:", len(commands_df))

Inventory loaded successfully!
{'pen': 20, 'book': 15, 'pencil': 30, 'notebook': 10, 'eraser': 25}

Command dataset loaded!
Total test commands: 6


In [3]:
# DFA-based command dataset testing

def test_command_dataset(commands_df):
    results = []

    for _, row in commands_df.iterrows():
        command = row["command_text"]

        # Identify the action from the command
        words = command.lower().split()

        if words[0] in ["add", "insert", "increase", "restock"]:
            detected_intent = "ADD"
        elif words[0] in ["remove", "delete", "subtract", "reduce"]:
            detected_intent = "REMOVE"
        elif words[0] in ["check", "show", "find", "get"]:
            detected_intent = "CHECK"
        elif words[0] in ["list", "display", "view"]:
            detected_intent = "LIST"
        else:
            detected_intent = "UNKNOWN"

        expected_intent = row["intent"]
        passed = detected_intent == expected_intent

        results.append({
            "Command": command,
            "Expected": expected_intent,
            "Detected": detected_intent,
            "Test Result": "PASS" if passed else "FAIL"
        })

    result_df = pd.DataFrame(results)
    display(result_df)

    accuracy = (
        (result_df["Test Result"] == "PASS").mean() * 100
    )
    print(f"Command intent detection accuracy: {accuracy:.2f}%")

test_command_dataset(commands_df)

,Command,Expected,Detected,Test Result
0,Add 10 pens,ADD,ADD,PASS
1,Remove 3 books,REMOVE,REMOVE,PASS
2,Check stock of pencils,CHECK,CHECK,PASS
3,List inventory,LIST,LIST,PASS
4,Add pens,ADD,ADD,PASS
5,Fly 5 pens,UNKNOWN,UNKNOWN,PASS


Command intent detection accuracy: 100.00%


In [4]:
# DFA State Transition Table

transition_data = [
    ["q0", "add / insert / restock", "q_ADD", "Action identified"],
    ["q0", "remove / delete", "q_REMOVE", "Action identified"],
    ["q0", "check / show / get", "q_CHECK", "Action identified"],
    ["q0", "list / display / view", "q_LIST", "Action identified"],
    ["q_ADD", "quantity and item", "q_ACCEPT", "Valid add command"],
    ["q_REMOVE", "quantity and item", "q_ACCEPT", "Valid remove command"],
    ["q_CHECK", "item name", "q_ACCEPT", "Valid stock query"],
    ["q_LIST", "end of command", "q_ACCEPT", "Valid list command"],
    ["q0", "unknown action", "q_REJECT", "Invalid command"],
    ["q_ADD", "missing quantity or item", "q_REJECT", "Incomplete command"],
    ["q_REMOVE", "missing quantity or item", "q_REJECT", "Incomplete command"]
]

transition_df = pd.DataFrame(
    transition_data,
    columns=["Current State", "Input", "Next State", "Meaning"]
)

print("DFA TRANSITION TABLE")
display(transition_df)

DFA TRANSITION TABLE


,Current State,Input,Next State,Meaning
0,q0,add / insert / restock,q_ADD,Action identified
1,q0,remove / delete,q_REMOVE,Action identified
2,q0,check / show / get,q_CHECK,Action identified
3,q0,list / display / view,q_LIST,Action identified
4,q_ADD,quantity and item,q_ACCEPT,Valid add command
5,q_REMOVE,quantity and item,q_ACCEPT,Valid remove command
6,q_CHECK,item name,q_ACCEPT,Valid stock query
7,q_LIST,end of command,q_ACCEPT,Valid list command
8,q0,unknown action,q_REJECT,Invalid command
9,q_ADD,missing quantity or item,q_REJECT,Incomplete command


In [5]:
# ACTUAL INVENTORY MANAGEMENT USING COMMANDS

import re
import pandas as pd
from IPython.display import display

# Load the inventory CSV if it is not already loaded
inventory_df = pd.read_csv("inventory_dataset.csv")

def execute_inventory_command(command):
    words = command.lower().strip().split()

    if not words:
        print("Invalid command!")
        return

    action = words[0]

    # LIST inventory
    if action in ["list", "display", "view"]:
        display(inventory_df)
        return

    # ADD or REMOVE command
    if action in ["add", "insert", "restock",
                  "remove", "delete", "subtract", "reduce"]:

        match = re.search(r"\b(\d+)\b", command)
        if not match:
            print("Rejected: Quantity is missing.")
            return

        quantity = int(match.group(1))
        if quantity <= 0:
            print("Rejected: Quantity must be positive.")
            return

        item_words = [
            w for w in words
            if w not in [
                action, "to", "the", "in", "into", "from",
                "inventory", "stock", "item", "items"
            ] and not w.isdigit()
        ]

        if not item_words:
            print("Rejected: Item name is missing.")
            return

        item = " ".join(item_words)
        if item.endswith("s") and not item.endswith("ss"):
            item = item[:-1]

        found = inventory_df["item_name"] == item

        if not found.any():
            print("Item not found:", item)
            return

        idx = inventory_df.index[found][0]
        current = int(inventory_df.loc[idx, "quantity"])

        if action in ["add", "insert", "restock"]:
            inventory_df.loc[idx, "quantity"] = current + quantity
            print("Accepted: Stock added successfully.")

        else:
            if quantity > current:
                print("Rejected: Insufficient stock.")
                return

            inventory_df.loc[idx, "quantity"] = current - quantity
            print("Accepted: Stock removed successfully.")

        # Save updated inventory
        inventory_df.to_csv("inventory_dataset.csv", index=False)
        print("Item:", item)
        print("Updated stock:",
              int(inventory_df.loc[idx, "quantity"]))

    # CHECK stock
    elif action in ["check", "show", "find", "get"]:
        item_words = [
            w for w in words
            if w not in [
                action, "stock", "of", "the", "inventory",
                "quantity", "for", "please", "item", "items"
            ]
        ]

        item = " ".join(item_words)
        if item.endswith("s") and not item.endswith("ss"):
            item = item[:-1]

        found = inventory_df["item_name"] == item

        if found.any():
            qty = int(inventory_df.loc[found, "quantity"].iloc[0])
            print(f"Current stock of {item}: {qty}")
        else:
            print("Item not found:", item)

    else:
        print("Rejected: Unknown command.")


# TEST THE COMMANDS
execute_inventory_command("Add 10 pens")
execute_inventory_command("Remove 3 books")
execute_inventory_command("Check stock of pencils")
execute_inventory_command("List inventory")

Accepted: Stock added successfully.
Item: pen
Updated stock: 30
Accepted: Stock removed successfully.
Item: book
Updated stock: 12
Current stock of pencil: 30


,item_id,item_name,category,quantity,unit_price
0,I001,pen,Stationery,30,10
1,I002,book,Stationery,12,50
2,I003,pencil,Stationery,30,5
3,I004,notebook,Stationery,10,40
4,I005,eraser,Stationery,25,5


In [ ]:
# Interactive Inventory Command Testing

print("AI INVENTORY MANAGEMENT SYSTEM")
print("Examples:")
print("Add 5 pens")
print("Remove 2 books")
print("Check stock of pencils")
print("List inventory")
print("Type exit to stop")

while True:
    command = input("\nEnter command: ")

    if command.lower().strip() == "exit":
        print("Program stopped.")
        break

    execute_inventory_command(command)

AI INVENTORY MANAGEMENT SYSTEM
Examples:
Add 5 pens
Remove 2 books
Check stock of pencils
List inventory
Type exit to stop
Current stock of pen: 35

Enter command: Add 5 pens
Accepted: Stock added successfully.
Item: pen
Updated stock: 40


In [7]:
print("AI INVENTORY MANAGEMENT SYSTEM")
print("=" * 40)

print("Total inventory items:", len(inventory_df))
print("Total test commands:", len(commands_df))
print("Total stock units:", inventory_df["quantity"].sum())

print("\nCurrent Inventory:")
display(inventory_df)

AI INVENTORY MANAGEMENT SYSTEM
Total inventory items: 5
Total test commands: 6
Total stock units: 110

Current Inventory:


,item_id,item_name,category,quantity,unit_price
0,I001,pen,Stationery,35,10
1,I002,book,Stationery,10,50
2,I003,pencil,Stationery,30,5
3,I004,notebook,Stationery,10,40
4,I005,eraser,Stationery,25,5
